In [0]:
%sql
CREATE TABLE IF NOT EXISTS target_catalog.control.ingestion_control
(
    table_id STRING,
    source_table STRING,
    target_table STRING,
    primary_key STRING,
    watermark_column STRING,
    last_processed_value TIMESTAMP,
    load_type STRING,
    active BOOLEAN,
    last_run_at TIMESTAMP,
    rows_processed BIGINT,
    status STRING,
    error_message STRING
)
USING DELTA;

In [0]:
MERGE INTO target_catalog.control.ingestion_control AS t

USING (
    SELECT * FROM VALUES
      ('customers',
       'source_catalog.raw.customers',
       'target_catalog.curated.customers',
       'customer_id', 'last_updated'),

      ('products',
       'source_catalog.raw.products',
       'target_catalog.curated.products',
       'product_id', 'last_updated'),

      ('orders',
       'source_catalog.raw.orders',
       'target_catalog.curated.orders',
       'order_id', 'last_updated'),

      ('employees',
       'source_catalog.raw.employees',
       'target_catalog.curated.employees',
       'employee_id', 'last_updated'),

      ('departments',
       'source_catalog.raw.departments',
       'target_catalog.curated.departments',
       'department_id', 'last_updated')

    AS config(
        table_id,
        source_table,
        target_table,
        primary_key,
        watermark_column
    )
) AS s

ON t.table_id = s.table_id

WHEN NOT MATCHED THEN
INSERT (
    table_id,
    source_table,
    target_table,
    primary_key,
    watermark_column,
    last_processed_value,
    load_type,
    active,
    last_run_at,
    rows_processed,
    status,
    error_message
)
VALUES (
    s.table_id,
    s.source_table,
    s.target_table,
    s.primary_key,
    s.watermark_column,
    TIMESTAMP '1900-01-01 00:00:00',
    'INCREMENTAL',
    TRUE,
    NULL,
    0,
    'NOT_RUN',
    NULL
);